# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (30-day readmissions) and role-based column masking.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')

# Mock Silver Data
mock_silver = pd.DataFrame({
    'person_id': ['pat_1', 'pat_1', 'pat_2', 'pat_3'],
    'condition_start_date': ['2023-01-01', '2023-01-15', '2023-01-01', '2023-01-01'],
    'condition_source_value': ['INDEX_CONDITION', 'FOLLOWUP', 'INDEX_CONDITION', 'INDEX_CONDITION']
})
mock_silver['condition_start_date'] = pd.to_datetime(mock_silver['condition_start_date'])
con.execute("CREATE TABLE dt_condition_occurrence AS SELECT * FROM mock_silver")


In [ ]:
# Simulate dbt Gold Mart logic (Readmissions)
gold_mart_query = """
WITH index_admissions AS (
    SELECT 
        person_id,
        condition_start_date AS index_date
    FROM dt_condition_occurrence
    WHERE condition_source_value = 'INDEX_CONDITION'
),
subsequent_admissions AS (
    SELECT 
        person_id,
        condition_start_date AS subsequent_date
    FROM dt_condition_occurrence
)
SELECT 
    i.person_id,
    i.index_date,
    COUNT(s.subsequent_date) AS readmission_count_30d
FROM index_admissions i
LEFT JOIN subsequent_admissions s 
    ON i.person_id = s.person_id 
    AND s.subsequent_date > i.index_date 
    AND (s.subsequent_date - i.index_date) <= 30
GROUP BY i.person_id, i.index_date
"""
print("Gold Mart - 30 Day Readmissions:")
print(con.execute(gold_mart_query).df())


In [ ]:
# Simulate Dynamic Masking
mock_patients = pd.DataFrame({
    'person_id': ['pat_1', 'pat_2'],
    'ssn': ['123-45-6789', '987-65-4321']
})
con.execute("CREATE TABLE patients AS SELECT * FROM mock_patients")

current_role = 'EHDIP_ANALYST' # Toggle to 'EHDIP_DATA_SCIENTIST' to see unmasked

masking_query = f"""
SELECT 
    person_id,
    CASE 
        WHEN '{current_role}' IN ('EHDIP_DATA_SCIENTIST', 'SYSADMIN') THEN ssn 
        ELSE '***MASKED***' 
    END as ssn
FROM patients
"""
print(f"\nMasked View for Role: {current_role}")
print(con.execute(masking_query).df())
